# 🥇 Gold Layer — Business Insights from Loan Transactions

**What we're doing:** Creating business-ready summaries from clean Silver data.

Gold tables power dashboards, executive reports, and KPIs.

In [ ]:
from pyspark.sql import functions as F

df_silver = spark.table('silver_loan_transactions')
print(f'Silver records available: {df_silver.count()}')

In [ ]:
# Gold: Monthly summary by Branch
df_gold = df_silver.groupBy('Branch', 'Month', 'TransactionType') \
    .agg(
        F.count('TransactionID').alias('TransactionCount'),
        F.round(F.sum('Amount'), 2).alias('TotalAmount'),
        F.round(F.avg('Amount'), 2).alias('AvgAmount')
    ).orderBy('Month', 'Branch')

df_gold.show(20)

In [ ]:
# Save Gold table
df_gold.write.format('delta').mode('overwrite').saveAsTable('gold_loan_summary')
print('✅ Gold table saved — ready for Power BI and dashboards!')

In [ ]:
%%sql
-- Executive view: Which branch processed the most loan volume?
SELECT Branch,
       SUM(TransactionCount) AS TotalTransactions,
       ROUND(SUM(TotalAmount), 2) AS TotalVolume
FROM gold_loan_summary
GROUP BY Branch
ORDER BY TotalVolume DESC

In [ ]:
%%sql
-- Trend: Monthly total loan payments vs disbursements
SELECT Month, TransactionType,
       SUM(TransactionCount) AS Transactions,
       ROUND(SUM(TotalAmount), 2) AS Volume
FROM gold_loan_summary
GROUP BY Month, TransactionType
ORDER BY Month